In [6]:
import pandas as pd
import sqlite3
import nflreadpy as nfl
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()
df_full['home_win'] = (df_full['home_score'] > df_full['away_score']).astype(int)

imputer = SimpleImputer(strategy='mean')
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

print(f"Loaded {len(df_full)} games, {len(df_full.columns)} columns")

Loaded 2458 games, 91 columns


In [13]:
feature_cols_v3 = [
    'home_recent_form', 'away_recent_form',
    'home_recent_point_diff', 'away_recent_point_diff',
    'home_qb_recent_yards', 'away_qb_recent_yards',
    'home_qb_recent_tds', 'away_qb_recent_tds',
    'home_qb_recent_ints', 'away_qb_recent_ints',
    'home_qb_recent_epa', 'away_qb_recent_epa',
    'home_rb_recent_rush_yards', 'away_rb_recent_rush_yards',
    'home_rb_recent_rush_epa', 'away_rb_recent_rush_epa',
    'home_rb_recent_rec_yards', 'away_rb_recent_rec_yards',
    'home_wrte_recent_rec_yards', 'away_wrte_recent_rec_yards',
    'home_wrte_recent_rec_epa', 'away_wrte_recent_rec_epa',
    'home_wrte_recent_targets', 'away_wrte_recent_targets',
    'home_qb_injury_flag', 'away_qb_injury_flag',
    'home_rb_injury_flag', 'away_rb_injury_flag',
    'home_wrte_injury_flag', 'away_wrte_injury_flag',
    'home_epa_allowed_recent', 'away_epa_allowed_recent',
    'home_yards_allowed_recent', 'away_yards_allowed_recent',
    'home_takeaways_recent', 'away_takeaways_recent',
    'home_coach_h2h_wins', 'h2h_games_played',
    'home_elo_pre', 'away_elo_pre',
    'div_game'
]

y = df_full['home_win']
train_mask = df_full['season'] <= 2021
test_mask = df_full['season'] >= 2022
y_train, y_test = y[train_mask], y[test_mask]

In [70]:
FEATURE_COLS = [
    'home_recent_form', 'away_recent_form',
    'home_recent_point_diff', 'away_recent_point_diff',
    'home_qb_recent_yards', 'away_qb_recent_yards',
    'home_qb_recent_tds', 'away_qb_recent_tds',
    'home_qb_recent_ints', 'away_qb_recent_ints',
    'home_qb_recent_epa', 'away_qb_recent_epa',
    'home_rb_recent_rush_yards', 'away_rb_recent_rush_yards',
    'home_rb_recent_rush_epa', 'away_rb_recent_rush_epa',
    'home_rb_recent_rec_yards', 'away_rb_recent_rec_yards',
    'home_wrte_recent_rec_yards', 'away_wrte_recent_rec_yards',
    'home_wrte_recent_rec_epa', 'away_wrte_recent_rec_epa',
    'home_wrte_recent_targets', 'away_wrte_recent_targets',
    'home_qb_injury_flag', 'away_qb_injury_flag',
    'home_rb_injury_flag', 'away_rb_injury_flag',
    'home_wrte_injury_flag', 'away_wrte_injury_flag',
    'home_epa_allowed_recent', 'away_epa_allowed_recent',
    'home_yards_allowed_recent', 'away_yards_allowed_recent',
    'home_takeaways_recent', 'away_takeaways_recent',
    'home_coach_h2h_wins', 'h2h_games_played',
    'home_elo_pre', 'away_elo_pre',
    'rest_advantage',
    'home_sack_rate_recent', 'away_sack_rate_recent',
    'home_pressure_pct_recent', 'away_pressure_pct_recent',
    'home_wr_height_advantage', 'away_wr_height_advantage',
    'home_wr_weight_advantage', 'away_wr_weight_advantage',
    'home_opp_cb_completion_allowed', 'away_opp_cb_completion_allowed',
    'home_opp_cb_rating_allowed', 'away_opp_cb_rating_allowed',
    'div_game'
]

climate_only_features = FEATURE_COLS + ['climate_shock', 'cold_shock_game']

y = df_full['home_win']
train_mask = df_full['season'] <= 2023
test_mask = df_full['season'] >= 2024
y_train, y_test = y[train_mask], y[test_mask]

imputer = SimpleImputer(strategy='mean')
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

for name, feats in [('Baseline (no weather)', FEATURE_COLS),
                     ('+ climate_shock + cold_shock_game only', climate_only_features)]:
    X = df_full[feats].copy()
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feats, index=X.index)
    X_train, X_test = X_imputed[train_mask], X_imputed[test_mask]
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"{name}: {acc:.3f}")

Baseline (no weather): 0.677
+ climate_shock + cold_shock_game only: 0.675


In [71]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()
df_full['home_win'] = (df_full['home_score'] > df_full['away_score']).astype(int)

# Reshape into one row per team per game (same pattern as add_recent_form_features)
home = df_full[['game_id', 'season', 'week', 'gameday', 'home_team_std', 'home_win', 'home_score', 'away_score']].copy()
home = home.rename(columns={'home_team_std': 'team'})
home['win'] = home['home_win']
home['point_diff'] = home['home_score'] - home['away_score']

away = df_full[['game_id', 'season', 'week', 'gameday', 'away_team_std', 'home_win', 'home_score', 'away_score']].copy()
away = away.rename(columns={'away_team_std': 'team'})
away['win'] = 1 - away['home_win']
away['point_diff'] = away['away_score'] - away['home_score']

team_games = pd.concat([home, away], ignore_index=True)
team_games = team_games.sort_values(['team', 'gameday']).reset_index(drop=True)

# For each team, get their most recent 5 games (their CURRENT form, no shift needed -
# these games have already happened, so this IS their state entering the next game)
current_form = (
    team_games.groupby('team')
    .apply(lambda g: g.tail(5)[['win', 'point_diff']].mean())
    .reset_index()
    .rename(columns={'win': 'current_recent_form', 'point_diff': 'current_recent_point_diff'})
)

current_form.sort_values('current_recent_form', ascending=False).head(10)

,team,current_recent_form,current_recent_point_diff
27,SEA,1.0,16.4
20,MIN,1.0,13.6
1,ATL,0.8,-3.0
14,JAX,0.8,15.8
21,NE,0.8,8.0
22,NO,0.8,7.2
12,HOU,0.8,5.2
3,BUF,0.6,5.8
16,LA,0.6,3.2
25,PHI,0.6,6.4


In [72]:
team_stats2 = df_full[['game_id', 'season', 'week', 'gameday', 'home_team_std', 'away_team_std', 'home_win']].copy()
team_stats2 = team_stats2.sort_values(['season', 'gameday']).reset_index(drop=True)

all_teams = set(team_stats2['home_team_std']).union(set(team_stats2['away_team_std']))
elo = {team: 1500 for team in all_teams}

k_factor = 20
home_advantage = 65
revert_fraction = 1/3
current_season = None

for idx, row in team_stats2.iterrows():
    if current_season is not None and row['season'] != current_season:
        for team in elo:
            elo[team] = elo[team] * (1 - revert_fraction) + 1500 * revert_fraction
    current_season = row['season']

    home_team = row['home_team_std']
    away_team = row['away_team_std']

    elo_diff = (elo[home_team] + home_advantage) - elo[away_team]
    expected_home = 1 / (1 + 10 ** (-elo_diff / 400))
    actual_home = row['home_win']

    elo[home_team] += k_factor * (actual_home - expected_home)
    elo[away_team] += k_factor * ((1 - actual_home) - (1 - expected_home))

current_elo = pd.DataFrame(list(elo.items()), columns=['team', 'current_elo'])
current_elo.sort_values('current_elo', ascending=False).head(10)

,team,current_elo
1,SEA,1632.675257
21,BUF,1601.453676
3,DEN,1589.710645
0,PHI,1588.229775
12,LA,1583.193187
13,HOU,1572.609956
6,DET,1564.548642
30,NE,1560.197268
22,SF,1559.509626
23,MIN,1551.110984


In [73]:
current_elo['current_elo'] = current_elo['current_elo'] * (1 - 1/3) + 1500 * (1/3)
current_elo.sort_values('current_elo', ascending=False).head(10)

,team,current_elo
1,SEA,1588.450171
21,BUF,1567.635784
3,DEN,1559.807097
0,PHI,1558.819850
12,LA,1555.462125
13,HOU,1548.406638
6,DET,1543.032428
30,NE,1540.131512
22,SF,1539.673084
23,MIN,1534.073990


In [74]:
sched_2026 = nfl.load_schedules(seasons=[2026]).to_pandas()
sched_2026[['game_id', 'week', 'gameday', 'home_team', 'away_team', 'home_rest', 'away_rest']].head(10)

,game_id,week,gameday,home_team,away_team,home_rest,away_rest
0,2026_01_NE_SEA,1,2026-09-09,SEA,NE,7,7
1,2026_01_SF_LA,1,2026-09-10,LA,SF,7,7
2,2026_01_CHI_CAR,1,2026-09-13,CAR,CHI,7,7
3,2026_01_TB_CIN,1,2026-09-13,CIN,TB,7,7
4,2026_01_NO_DET,1,2026-09-13,DET,NO,7,7
5,2026_01_BUF_HOU,1,2026-09-13,HOU,BUF,7,7
6,2026_01_BAL_IND,1,2026-09-13,IND,BAL,7,7
7,2026_01_CLE_JAX,1,2026-09-13,JAX,CLE,7,7
8,2026_01_ATL_PIT,1,2026-09-13,PIT,ATL,7,7
9,2026_01_NYJ_TEN,1,2026-09-13,TEN,NYJ,7,7


In [75]:
print([c for c in player_stats.columns if 'date' in c.lower() or 'game' in c.lower()])

['game_id']


In [77]:
player_stats = nfl.load_player_stats(seasons=list(range(2015, 2026))).to_pandas()

qb_stats = player_stats[(player_stats['position'] == 'QB') & (player_stats['attempts'] > 0)].copy()
qb_stats = qb_stats[['player_id', 'player_name', 'game_id', 'season', 'week', 'team',
                      'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_epa']]

# Attach real dates from df_full so we can sort chronologically (season/week alone can be
# ambiguous across playoff weeks)
game_dates = df_full[['game_id', 'gameday']].drop_duplicates()
qb_stats = qb_stats.merge(game_dates, on='game_id', how='left')

qb_stats = qb_stats.sort_values(['team', 'gameday']).reset_index(drop=True)

# Most recent starter per team = whoever has the LATEST game date for that team
current_qb = qb_stats.loc[qb_stats.groupby('team')['gameday'].idxmax()][['team', 'player_id', 'player_name', 'gameday']]

print(len(current_qb))
current_qb.sort_values('team').head(10)

32


,team,player_id,player_name,gameday
212,ARI,00-0033119,J.Brissett,2026-01-04
415,ATL,00-0029604,K.Cousins,2026-01-04
641,BAL,00-0034796,L.Jackson,2026-01-04
869,BUF,00-0034857,J.Allen,2026-01-17
1086,CAR,00-0039150,B.Young,2026-01-10
1293,CHI,00-0039918,C.Williams,2026-01-18
1505,CIN,00-0036442,J.Burrow,2026-01-04
1729,CLE,00-0040668,S.Sanders,2026-01-04
1949,DAL,00-0033077,D.Prescott,2026-01-04
2150,DEN,00-0035264,J.Stidham,2026-01-25


In [78]:
qb_stats['recent_passing_yards'] = (
    qb_stats.groupby('player_id')['passing_yards']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)
qb_stats['recent_passing_tds'] = (
    qb_stats.groupby('player_id')['passing_tds']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)
qb_stats['recent_passing_interceptions'] = (
    qb_stats.groupby('player_id')['passing_interceptions']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)
qb_stats['recent_passing_epa'] = (
    qb_stats.groupby('player_id')['passing_epa']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

# Get each QB's MOST RECENT rolling value (their current state)
current_qb_full = qb_stats.loc[qb_stats.groupby('team')['gameday'].idxmax()][
    ['team', 'player_id', 'player_name', 'recent_passing_yards', 'recent_passing_tds',
     'recent_passing_interceptions', 'recent_passing_epa']
]

current_qb_full.sort_values('team').head(10)

,team,player_id,player_name,recent_passing_yards,recent_passing_tds,recent_passing_interceptions,recent_passing_epa
212,ARI,00-0033119,J.Brissett,235.600000,2.0,0.800000,-2.265752
415,ATL,00-0029604,K.Cousins,207.600000,1.4,0.800000,1.867292
641,BAL,00-0034796,L.Jackson,190.800000,1.2,0.800000,-0.375927
869,BUF,00-0034857,J.Allen,228.200000,1.4,0.400000,1.259997
1086,CAR,00-0039150,B.Young,187.600000,1.2,0.600000,-2.228752
1293,CHI,00-0039918,C.Williams,282.000000,2.0,1.200000,7.180300
1505,CIN,00-0036442,J.Burrow,271.800000,2.6,1.000000,6.120583
1729,CLE,00-0040668,S.Sanders,199.000000,1.0,1.600000,-8.585892
1949,DAL,00-0033077,D.Prescott,258.200000,1.0,0.400000,5.085658
2150,DEN,00-0035264,J.Stidham,209.666667,1.0,0.666667,-3.493516


In [79]:
snaps_full = nfl.load_snap_counts(seasons=list(range(2015, 2026))).to_pandas()

rb_stats_live = player_stats[player_stats['position'] == 'RB'].copy()
rb_stats_live = rb_stats_live[['player_id', 'player_name', 'game_id', 'season', 'week', 'team',
                                'rushing_yards', 'rushing_epa', 'receiving_yards']]
rb_stats_live = rb_stats_live.merge(crosswalk, on='player_id', how='left')
rb_stats_live = rb_stats_live.merge(
    snaps_full[snaps_full['position'] == 'RB'][['pfr_player_id', 'game_id', 'offense_pct']],
    on=['pfr_player_id', 'game_id'], how='left'
)
rb_stats_live = rb_stats_live.merge(game_dates, on='game_id', how='left')
rb_stats_live = rb_stats_live.sort_values(['player_id', 'gameday']).reset_index(drop=True)

rb_stats_live['recent_snap_pct'] = (
    rb_stats_live.groupby('player_id')['offense_pct']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)
rb_stats_live['weighted_rush_yards'] = rb_stats_live['rushing_yards'] * rb_stats_live['offense_pct']
rb_stats_live['weighted_rush_epa'] = rb_stats_live['rushing_epa'] * rb_stats_live['offense_pct']
rb_stats_live['weighted_rec_yards'] = rb_stats_live['receiving_yards'] * rb_stats_live['offense_pct']

for col in ['weighted_rush_yards', 'weighted_rush_epa', 'weighted_rec_yards']:
    rb_stats_live[f'recent_{col}'] = (
        rb_stats_live.groupby('team')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

# Team-level current snapshot = most recent row per team (aggregated across RBs naturally
# via the team-level rolling above)
current_rb = rb_stats_live.loc[rb_stats_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_weighted_rush_yards', 'recent_weighted_rush_epa', 'recent_weighted_rec_yards']
]

current_rb.sort_values('team').head(10)

,team,recent_weighted_rush_yards,recent_weighted_rush_epa,recent_weighted_rec_yards
14120,ARI,21.688,0.417534,11.446
14815,ATL,12.424,-0.501018,2.316
6401,BAL,73.924,2.505956,0.864
14632,BUF,41.238,-0.437203,5.882
12557,CAR,18.446,-1.051942,5.260
12990,CHI,32.216,0.178368,8.998
7748,CIN,NaN,NaN,NaN
11506,CLE,2.076,0.195984,3.090
15974,DAL,3.288,-0.008795,2.124
14367,DEN,0.164,-0.115989,1.198


In [80]:
cin_check = rb_stats_live[rb_stats_live['team'] == 'CIN'].sort_values('gameday').tail(10)
cin_check[['player_id', 'player_name', 'gameday', 'offense_pct', 'rushing_yards', 'weighted_rush_yards', 'recent_weighted_rush_yards']]

,player_id,player_name,gameday,offense_pct,rushing_yards,weighted_rush_yards,recent_weighted_rush_yards
15801,00-0038597,C.Brown,2025-12-14,NaN,53,NaN,NaN
16896,00-0040208,T.Brooks,2025-12-14,NaN,0,NaN,NaN
7746,00-0033526,S.Perine,2025-12-21,NaN,25,NaN,NaN
15802,00-0038597,C.Brown,2025-12-21,NaN,66,NaN,NaN
16897,00-0040208,T.Brooks,2025-12-21,NaN,7,NaN,NaN
7747,00-0033526,S.Perine,2025-12-28,NaN,5,NaN,NaN
15803,00-0038597,C.Brown,2025-12-28,NaN,101,NaN,NaN
16898,00-0040208,T.Brooks,2025-12-28,NaN,10,NaN,NaN
7748,00-0033526,S.Perine,2026-01-04,NaN,42,NaN,NaN
15804,00-0038597,C.Brown,2026-01-04,NaN,72,NaN,NaN


In [81]:
recent_games = rb_stats_live[rb_stats_live['gameday'] >= '2025-11-01']
missing_pct = recent_games['offense_pct'].isnull().mean()
print(f"Missing offense_pct in recent (Nov 2025+) RB rows: {missing_pct:.1%}")

# Which teams are affected?
affected_teams = recent_games[recent_games['offense_pct'].isnull()]['team'].unique()
print(f"Teams affected: {sorted(affected_teams)}")

Missing offense_pct in recent (Nov 2025+) RB rows: 5.1%
Teams affected: ['BAL', 'BUF', 'CIN', 'SEA']


In [82]:
rb_stats_live = rb_stats_live.sort_values(['player_id', 'gameday']).reset_index(drop=True)
rb_stats_live['offense_pct'] = rb_stats_live.groupby('player_id')['offense_pct'].ffill()

# Recalculate everything downstream with the filled values
rb_stats_live['weighted_rush_yards'] = rb_stats_live['rushing_yards'] * rb_stats_live['offense_pct']
rb_stats_live['weighted_rush_epa'] = rb_stats_live['rushing_epa'] * rb_stats_live['offense_pct']
rb_stats_live['weighted_rec_yards'] = rb_stats_live['receiving_yards'] * rb_stats_live['offense_pct']

for col in ['weighted_rush_yards', 'weighted_rush_epa', 'weighted_rec_yards']:
    rb_stats_live[f'recent_{col}'] = (
        rb_stats_live.groupby('team')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

current_rb = rb_stats_live.loc[rb_stats_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_weighted_rush_yards', 'recent_weighted_rush_epa', 'recent_weighted_rec_yards']
]

current_rb[current_rb['team'].isin(['BAL', 'BUF', 'CIN', 'SEA'])]

,team,recent_weighted_rush_yards,recent_weighted_rush_epa,recent_weighted_rec_yards
6401,BAL,73.924,2.505956,0.864
14632,BUF,41.238,-0.437203,5.882
7748,CIN,11.310,-0.070930,3.276
15530,SEA,52.688,0.327379,16.738


In [83]:
wrte_stats_live = player_stats[player_stats['position'].isin(['WR', 'TE'])].copy()
wrte_stats_live = wrte_stats_live[['player_id', 'player_name', 'game_id', 'season', 'week', 'team',
                                    'targets', 'receiving_yards', 'receiving_epa']]
wrte_stats_live = wrte_stats_live.merge(crosswalk, on='player_id', how='left')
wrte_stats_live = wrte_stats_live.merge(
    snaps_full[snaps_full['position'].isin(['WR', 'TE'])][['pfr_player_id', 'game_id', 'offense_pct']],
    on=['pfr_player_id', 'game_id'], how='left'
)
wrte_stats_live = wrte_stats_live.merge(game_dates, on='game_id', how='left')
wrte_stats_live = wrte_stats_live.sort_values(['player_id', 'gameday']).reset_index(drop=True)

# Forward-fill snap share gaps proactively, same fix as RB
wrte_stats_live['offense_pct'] = wrte_stats_live.groupby('player_id')['offense_pct'].ffill()

wrte_stats_live['weighted_rec_yards'] = wrte_stats_live['receiving_yards'] * wrte_stats_live['offense_pct']
wrte_stats_live['weighted_rec_epa'] = wrte_stats_live['receiving_epa'] * wrte_stats_live['offense_pct']
wrte_stats_live['weighted_targets'] = wrte_stats_live['targets'] * wrte_stats_live['offense_pct']

for col in ['weighted_rec_yards', 'weighted_rec_epa', 'weighted_targets']:
    wrte_stats_live[f'recent_{col}'] = (
        wrte_stats_live.groupby('team')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

current_wrte = wrte_stats_live.loc[wrte_stats_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_weighted_rec_yards', 'recent_weighted_rec_epa', 'recent_weighted_targets']
]

print(current_wrte['recent_weighted_rec_yards'].isnull().sum())
current_wrte.sort_values('team').head(10)

0


,team,recent_weighted_rec_yards,recent_weighted_rec_epa,recent_weighted_targets
20127,ARI,0.034,-0.150514,0.068
28193,ATL,17.336,-0.916067,2.978
9182,BAL,5.000,0.045319,0.606
10659,BUF,18.190,0.825793,2.040
34654,CAR,13.118,0.831454,1.838
26154,CHI,1.020,0.081289,0.184
26409,CIN,14.988,1.424461,1.360
32103,CLE,32.496,0.277636,4.604
31863,DAL,47.360,1.920814,5.490
22042,DEN,9.892,-0.527222,1.392


In [84]:
ari_check = wrte_stats_live[wrte_stats_live['team'] == 'ARI'].sort_values('gameday').tail(10)
ari_check[['player_id', 'player_name', 'gameday', 'offense_pct', 'receiving_yards', 'weighted_rec_yards']]

,player_id,player_name,gameday,offense_pct,receiving_yards,weighted_rec_yards
38418,00-0039041,E.Higgins,2025-12-28,0.60,0,0.00
39170,00-0039521,X.Weaver,2025-12-28,0.67,24,16.08
39493,00-0039849,M.Harrison,2025-12-28,0.33,0,0.00
20127,00-0033439,P.Brown,2026-01-04,0.22,0,0.00
26823,00-0034928,S.Sims,2026-01-04,0.04,0,0.00
31675,00-0036332,J.Deguara,2026-01-04,0.11,15,1.65
36082,00-0037744,T.McBride,2026-01-04,0.95,65,61.75
37327,00-0038559,M.Wilson,2026-01-04,0.93,99,92.07
37746,00-0038640,J.Brooks,2026-01-04,0.27,0,0.00
38419,00-0039041,E.Higgins,2026-01-04,0.60,28,16.80


In [85]:
wrte_team_game_live = wrte_stats_live.groupby(['team', 'game_id', 'gameday'], as_index=False).agg(
    team_weighted_rec_yards=('weighted_rec_yards', 'sum'),
    team_weighted_rec_epa=('weighted_rec_epa', 'sum'),
    team_weighted_targets=('weighted_targets', 'sum')
)

wrte_team_game_live = wrte_team_game_live.sort_values(['team', 'gameday']).reset_index(drop=True)

for col in ['team_weighted_rec_yards', 'team_weighted_rec_epa', 'team_weighted_targets']:
    wrte_team_game_live[f'recent_{col}'] = (
        wrte_team_game_live.groupby('team')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

current_wrte = wrte_team_game_live.loc[wrte_team_game_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_team_weighted_rec_yards', 'recent_team_weighted_rec_epa', 'recent_team_weighted_targets']
]

print(current_wrte['recent_team_weighted_rec_yards'].isnull().sum())
current_wrte.sort_values('team').head(10)

0


,team,recent_team_weighted_rec_yards,recent_team_weighted_rec_epa,recent_team_weighted_targets
183,ARI,169.974,5.680289,22.018
369,ATL,131.914,3.366489,18.490
559,BAL,117.006,6.362094,11.154
755,BUF,110.644,4.669646,14.630
941,CAR,126.574,3.931692,15.330
1126,CHI,174.212,6.432767,22.598
1314,CIN,161.086,4.425208,19.316
1498,CLE,106.864,-0.777953,14.370
1686,DAL,165.846,6.310962,19.844
1873,DEN,102.440,0.789096,16.244


In [86]:
current_rb_check = rb_stats_live[rb_stats_live['team'] == 'BAL'].sort_values('gameday').tail(10)
current_rb_check[['player_id', 'player_name', 'gameday', 'offense_pct', 'rushing_yards', 'weighted_rush_yards', 'recent_weighted_rush_yards']]

,player_id,player_name,gameday,offense_pct,rushing_yards,weighted_rush_yards,recent_weighted_rush_yards
15603,00-0038454,K.Mitchell,2025-12-21,0.35,13,4.55,9.212
16679,00-0039796,R.Ali,2025-12-21,0.22,0,0.00,1.768
6400,00-0032764,D.Henry,2025-12-27,0.63,216,136.08,60.832
7621,00-0033376,P.Ricard,2025-12-27,NaN,0,NaN,NaN
15604,00-0038454,K.Mitchell,2025-12-27,0.18,31,5.58,9.948
16680,00-0039796,R.Ali,2025-12-27,0.20,0,0.00,1.768
6401,00-0032764,D.Henry,2026-01-04,0.71,126,89.46,73.924
7622,00-0033376,P.Ricard,2026-01-04,NaN,0,NaN,NaN
15605,00-0038454,K.Mitchell,2026-01-04,0.14,2,0.28,9.510
16681,00-0039796,R.Ali,2026-01-04,0.16,0,0.00,0.136


In [87]:
rb_team_game_live = rb_stats_live.groupby(['team', 'game_id', 'gameday'], as_index=False).agg(
    team_weighted_rush_yards=('weighted_rush_yards', 'sum'),
    team_weighted_rush_epa=('weighted_rush_epa', 'sum'),
    team_weighted_rec_yards=('weighted_rec_yards', 'sum')
)

In [88]:
rb_team_game_live = rb_team_game_live.sort_values(['team', 'gameday']).reset_index(drop=True)

for col in ['team_weighted_rush_yards', 'team_weighted_rush_epa', 'team_weighted_rec_yards']:
    rb_team_game_live[f'recent_{col}'] = (
        rb_team_game_live.groupby('team')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

current_rb = rb_team_game_live.loc[rb_team_game_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_team_weighted_rush_yards', 'recent_team_weighted_rush_epa', 'recent_team_weighted_rec_yards']
]

print(current_rb['recent_team_weighted_rush_yards'].isnull().sum())
current_rb[current_rb['team'] == 'BAL']

0


,team,recent_team_weighted_rush_yards,recent_team_weighted_rush_epa,recent_team_weighted_rec_yards
559,BAL,83.57,2.397753,4.0


In [89]:
current_coach = df_full.loc[df_full.groupby('home_team_std')['gameday'].idxmax()][['home_team_std', 'home_coach']].rename(
    columns={'home_team_std': 'team', 'home_coach': 'coach'})

# Some teams' most recent game might have been an away game - check both sides and take the truly latest
current_coach_away = df_full.loc[df_full.groupby('away_team_std')['gameday'].idxmax()][['away_team_std', 'away_coach', 'gameday']].rename(
    columns={'away_team_std': 'team', 'away_coach': 'coach'})

current_coach_home = df_full.loc[df_full.groupby('home_team_std')['gameday'].idxmax()][['home_team_std', 'home_coach', 'gameday']].rename(
    columns={'home_team_std': 'team', 'home_coach': 'coach'})

all_coach_appearances = pd.concat([current_coach_home, current_coach_away], ignore_index=True)
current_coach = all_coach_appearances.loc[all_coach_appearances.groupby('team')['gameday'].idxmax()][['team', 'coach']]

current_coach.sort_values('team').head(10)

,team,coach
32,ARI,Jonathan Gannon
1,ATL,Raheem Morris
34,BAL,John Harbaugh
35,BUF,Sean McDermott
4,CAR,Dave Canales
5,CHI,Ben Johnson
6,CIN,Zac Taylor
39,CLE,Kevin Stefanski
40,DAL,Brian Schottenheimer
9,DEN,Sean Payton


In [90]:
# --- Defense-allowed (EPA/yards allowed, takeaways) ---
team_stats_live = nfl.load_team_stats(seasons=list(range(2015, 2026))).to_pandas()
team_stats_live = team_stats_live[['game_id', 'season', 'week', 'team', 'opponent_team',
                                     'passing_epa', 'rushing_epa', 'passing_yards', 'rushing_yards',
                                     'def_interceptions', 'fumble_recovery_opp', 'attempts', 'sacks_suffered']]

opponent_offense_live = team_stats_live[['game_id', 'team', 'passing_epa', 'rushing_epa',
                                           'passing_yards', 'rushing_yards']].rename(
    columns={'team': 'opponent_team', 'passing_epa': 'opp_passing_epa', 'rushing_epa': 'opp_rushing_epa',
             'passing_yards': 'opp_passing_yards', 'rushing_yards': 'opp_rushing_yards'})

team_stats_live = team_stats_live.merge(opponent_offense_live, on=['game_id', 'opponent_team'], how='left')
team_stats_live['epa_allowed'] = team_stats_live['opp_passing_epa'] + team_stats_live['opp_rushing_epa']
team_stats_live['yards_allowed'] = team_stats_live['opp_passing_yards'] + team_stats_live['opp_rushing_yards']
team_stats_live['takeaways'] = team_stats_live['def_interceptions'] + team_stats_live['fumble_recovery_opp']
team_stats_live['sack_rate'] = team_stats_live['sacks_suffered'] / (team_stats_live['attempts'] + team_stats_live['sacks_suffered'])

team_stats_live = team_stats_live.merge(game_dates, on='game_id', how='left')
team_stats_live = team_stats_live.sort_values(['team', 'gameday']).reset_index(drop=True)

for col in ['epa_allowed', 'yards_allowed', 'takeaways', 'sack_rate']:
    team_stats_live[f'recent_{col}'] = (
        team_stats_live.groupby('team')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

current_team_stats = team_stats_live.loc[team_stats_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_epa_allowed', 'recent_yards_allowed', 'recent_takeaways', 'recent_sack_rate']
]

# --- Pressure rate (raw team codes, PFR convention) ---
pressure_stats_live = nfl.load_pfr_advstats(seasons=list(range(2018, 2026)), stat_type='pass').to_pandas()
pressure_stats_live = pressure_stats_live[['game_id', 'season', 'week', 'team', 'times_pressured_pct']]
pressure_stats_live = pressure_stats_live.merge(game_dates, on='game_id', how='left')
pressure_stats_live = pressure_stats_live.sort_values(['team', 'gameday']).reset_index(drop=True)

pressure_stats_live['recent_pressure_pct'] = (
    pressure_stats_live.groupby('team')['times_pressured_pct']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

current_pressure = pressure_stats_live.loc[pressure_stats_live.groupby('team')['gameday'].idxmax()][
    ['team', 'recent_pressure_pct']
]

print(current_team_stats['recent_epa_allowed'].isnull().sum())
print(current_pressure['recent_pressure_pct'].isnull().sum())
current_team_stats.sort_values('team').head(10)

0
0


,team,recent_epa_allowed,recent_yards_allowed,recent_takeaways,recent_sack_rate
183,ARI,17.360788,426.6,0.4,0.085502
369,ATL,0.750808,365.8,1.6,0.038674
559,BAL,2.031505,374.2,1.2,0.108430
755,BUF,-4.633861,269.6,1.0,0.063889
941,CAR,2.414398,348.0,1.0,0.044673
1126,CHI,10.411571,427.4,0.6,0.009964
1314,CIN,2.208267,329.0,1.2,0.056260
1498,CLE,-2.563238,331.2,0.6,0.111130
1686,DAL,15.004925,384.8,0.4,0.063198
1873,DEN,-6.762826,290.0,1.4,0.056572


In [91]:
wr_snaps_live = player_stats[player_stats['position'] == 'WR'][['player_id', 'game_id', 'season', 'week', 'team']].copy()
wr_snaps_live = wr_snaps_live.merge(crosswalk, on='player_id', how='left')
wr_snaps_live = wr_snaps_live.merge(
    snaps_full[snaps_full['position'] == 'WR'][['pfr_player_id', 'game_id', 'offense_pct']],
    on=['pfr_player_id', 'game_id'], how='left'
)
wr_snaps_live = wr_snaps_live.merge(game_dates, on='game_id', how='left')
wr_snaps_live = wr_snaps_live.sort_values(['player_id', 'gameday']).reset_index(drop=True)
wr_snaps_live['offense_pct'] = wr_snaps_live.groupby('player_id')['offense_pct'].ffill()

wr_snaps_live['recent_offense_pct'] = (
    wr_snaps_live.groupby('player_id')['offense_pct']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

# Each player's own most recent appearance = their CURRENT recent snap share
each_wr_current = wr_snaps_live.loc[wr_snaps_live.groupby('player_id')['gameday'].idxmax()][
    ['player_id', 'pfr_player_id', 'team', 'recent_offense_pct', 'gameday']
]

# Now pick the highest current recent_offense_pct PER TEAM (their current WR1)
current_primary_wr = each_wr_current.loc[each_wr_current.groupby('team')['recent_offense_pct'].idxmax()][
    ['team', 'pfr_player_id', 'recent_offense_pct']
]

current_primary_wr = current_primary_wr.merge(physical, on='pfr_player_id', how='left')
current_primary_wr.sort_values('team').head(10)

,team,pfr_player_id,recent_offense_pct,height,weight
0,ARI,WilsMi02,0.858,74.0,213.0
1,ATL,LondDr00,0.902,76.0,215.0
2,BAL,FlowZa00,0.850,69.0,183.0
3,BUF,HarvPe00,0.744,71.0,184.0
4,CAR,McMiTe00,0.864,76.0,219.0
5,CHI,MoorD.00,0.860,72.0,213.0
6,CIN,ChasJa00,0.922,72.0,205.0
7,CLE,JeudJe00,0.878,73.0,195.0
8,DAL,PickGe00,0.780,75.0,205.0
9,DEN,SuttCo00,0.864,76.0,216.0


In [92]:
cb_snaps_live = adv_def_full.copy()  # from earlier: game_id, season, week, team, pfr_player_id, completion%, rating allowed

cb_only_snaps = snaps_full[snaps_full['position'] == 'CB'][['pfr_player_id', 'game_id', 'defense_pct']]

# INNER merge - critical fix from the earlier bug, applied proactively here
cb_stats_live = cb_snaps_live.merge(cb_only_snaps, on=['pfr_player_id', 'game_id'], how='inner')
cb_stats_live = cb_stats_live.merge(game_dates, on='game_id', how='left')
cb_stats_live = cb_stats_live.sort_values(['pfr_player_id', 'gameday']).reset_index(drop=True)

for col in ['defense_pct', 'def_completion_pct', 'def_passer_rating_allowed']:
    cb_stats_live[f'recent_{col}'] = (
        cb_stats_live.groupby('pfr_player_id')[col]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
    )

each_cb_current = cb_stats_live.loc[cb_stats_live.groupby('pfr_player_id')['gameday'].idxmax()][
    ['team', 'pfr_player_id', 'recent_defense_pct', 'recent_def_completion_pct', 'recent_def_passer_rating_allowed']
]

current_primary_cb = each_cb_current.loc[each_cb_current.groupby('team')['recent_defense_pct'].idxmax()][
    ['team', 'pfr_player_id', 'recent_def_completion_pct', 'recent_def_passer_rating_allowed']
]

current_primary_cb = current_primary_cb.merge(physical, on='pfr_player_id', how='left')
current_primary_cb.sort_values('team').head(10)

,team,pfr_player_id,recent_def_completion_pct,recent_def_passer_rating_allowed,height,weight
0,ARI,JohnWi02,0.7094,109.62,74.0,200.0
1,ATL,TerrAJ00,0.5078,86.90,73.0,200.0
2,BAL,WiggNa00,0.6450,84.48,73.0,182.0
3,BUF,BenfCh00,0.4200,64.00,73.0,205.0
4,CAR,JackMi01,0.5678,65.78,73.0,210.0
5,CHI,JohnJa13,0.7500,96.78,72.0,195.0
6,CIN,WebbBW00,0.4846,83.08,71.0,190.0
7,CLE,CampTy01,0.4866,78.58,73.0,195.0
8,DAL,BlanDa00,0.6032,108.32,74.0,200.0
9,DEN,JackKa99,0.5906,63.18,70.0,183.0


In [93]:
home_team, away_team = 'SEA', 'NE'

# Coach head-to-head - compute over ALL real historical meetings (not tied to one training row)
home_coach_name = current_coach[current_coach['team'] == home_team]['coach'].values[0]
away_coach_name = current_coach[current_coach['team'] == away_team]['coach'].values[0]

coach_meetings = df_full[
    ((df_full['home_coach'] == home_coach_name) & (df_full['away_coach'] == away_coach_name)) |
    ((df_full['home_coach'] == away_coach_name) & (df_full['away_coach'] == home_coach_name))
]

if len(coach_meetings) == 0:
    home_coach_h2h_wins, h2h_games_played = 0.5, 0
else:
    home_coach_wins = (
        ((coach_meetings['home_coach'] == home_coach_name) & (coach_meetings['home_win'] == 1)) |
        ((coach_meetings['away_coach'] == home_coach_name) & (coach_meetings['home_win'] == 0))
    ).sum()
    home_coach_h2h_wins = home_coach_wins / len(coach_meetings)
    h2h_games_played = len(coach_meetings)

print(f"Coaches: {home_coach_name} (home) vs {away_coach_name} (away)")
print(f"H2H: {home_coach_h2h_wins:.3f} over {h2h_games_played} meetings")

Coaches: Mike Macdonald (home) vs Mike Vrabel (away)
H2H: 1.000 over 1 meetings


In [94]:
row = {}

# Team form + point differential
row['home_recent_form'] = current_form[current_form['team']==home_team]['current_recent_form'].values[0]
row['away_recent_form'] = current_form[current_form['team']==away_team]['current_recent_form'].values[0]
row['home_recent_point_diff'] = current_form[current_form['team']==home_team]['current_recent_point_diff'].values[0]
row['away_recent_point_diff'] = current_form[current_form['team']==away_team]['current_recent_point_diff'].values[0]

# QB
for side, team in [('home', home_team), ('away', away_team)]:
    qb_row = current_qb_full[current_qb_full['team']==team]
    row[f'{side}_qb_recent_yards'] = qb_row['recent_passing_yards'].values[0]
    row[f'{side}_qb_recent_tds'] = qb_row['recent_passing_tds'].values[0]
    row[f'{side}_qb_recent_ints'] = qb_row['recent_passing_interceptions'].values[0]
    row[f'{side}_qb_recent_epa'] = qb_row['recent_passing_epa'].values[0]

# RB
for side, team in [('home', home_team), ('away', away_team)]:
    rb_row = current_rb[current_rb['team']==team]
    row[f'{side}_rb_recent_rush_yards'] = rb_row['recent_team_weighted_rush_yards'].values[0]
    row[f'{side}_rb_recent_rush_epa'] = rb_row['recent_team_weighted_rush_epa'].values[0]
    row[f'{side}_rb_recent_rec_yards'] = rb_row['recent_team_weighted_rec_yards'].values[0]

# WR/TE
for side, team in [('home', home_team), ('away', away_team)]:
    wrte_row = current_wrte[current_wrte['team']==team]
    row[f'{side}_wrte_recent_rec_yards'] = wrte_row['recent_team_weighted_rec_yards'].values[0]
    row[f'{side}_wrte_recent_rec_epa'] = wrte_row['recent_team_weighted_rec_epa'].values[0]
    row[f'{side}_wrte_recent_targets'] = wrte_row['recent_team_weighted_targets'].values[0]

# Injuries - default healthy (0) since no game-week injury report exists yet this far out
for side in ['home', 'away']:
    row[f'{side}_qb_injury_flag'] = 0
    row[f'{side}_rb_injury_flag'] = 0
    row[f'{side}_wrte_injury_flag'] = 0

# Defense-allowed, sack rate
for side, team in [('home', home_team), ('away', away_team)]:
    ts_row = current_team_stats[current_team_stats['team']==team]
    row[f'{side}_epa_allowed_recent'] = ts_row['recent_epa_allowed'].values[0]
    row[f'{side}_yards_allowed_recent'] = ts_row['recent_yards_allowed'].values[0]
    row[f'{side}_takeaways_recent'] = ts_row['recent_takeaways'].values[0]
    row[f'{side}_sack_rate_recent'] = ts_row['recent_sack_rate'].values[0]

# Pressure rate (raw team codes - SEA/NE never relocated, so same either way)
for side, team in [('home', home_team), ('away', away_team)]:
    p_row = current_pressure[current_pressure['team']==team]
    row[f'{side}_pressure_pct_recent'] = p_row['recent_pressure_pct'].values[0] if len(p_row) > 0 else None

# Coach h2h
row['home_coach_h2h_wins'] = home_coach_h2h_wins
row['h2h_games_played'] = h2h_games_played

# Elo
row['home_elo_pre'] = current_elo[current_elo['team']==home_team]['current_elo'].values[0]
row['away_elo_pre'] = current_elo[current_elo['team']==away_team]['current_elo'].values[0]

# Rest advantage - from the 2026 schedule
game_row = sched_2026[sched_2026['game_id']=='2026_01_NE_SEA']
row['rest_advantage'] = game_row['home_rest'].values[0] - game_row['away_rest'].values[0]

# DB/WR matchup: home team's WR vs away team's CB, and vice versa
home_wr = current_primary_wr[current_primary_wr['team']==home_team]
away_cb = current_primary_cb[current_primary_cb['team']==away_team]
row['home_wr_height_advantage'] = home_wr['height'].values[0] - away_cb['height'].values[0]
row['home_wr_weight_advantage'] = home_wr['weight'].values[0] - away_cb['weight'].values[0]
row['home_opp_cb_completion_allowed'] = away_cb['recent_def_completion_pct'].values[0]
row['home_opp_cb_rating_allowed'] = away_cb['recent_def_passer_rating_allowed'].values[0]

away_wr = current_primary_wr[current_primary_wr['team']==away_team]
home_cb = current_primary_cb[current_primary_cb['team']==home_team]
row['away_wr_height_advantage'] = away_wr['height'].values[0] - home_cb['height'].values[0]
row['away_wr_weight_advantage'] = away_wr['weight'].values[0] - home_cb['weight'].values[0]
row['away_opp_cb_completion_allowed'] = home_cb['recent_def_completion_pct'].values[0]
row['away_opp_cb_rating_allowed'] = home_cb['recent_def_passer_rating_allowed'].values[0]

# Div game
row['div_game'] = game_row['div_game'].values[0]

feature_row = pd.DataFrame([row])
feature_row

,home_recent_form,away_recent_form,home_recent_point_diff,away_recent_point_diff,home_qb_recent_yards,home_qb_recent_tds,home_qb_recent_ints,home_qb_recent_epa,away_qb_recent_yards,away_qb_recent_tds,...,rest_advantage,home_wr_height_advantage,home_wr_weight_advantage,home_opp_cb_completion_allowed,home_opp_cb_rating_allowed,away_wr_height_advantage,away_wr_weight_advantage,away_opp_cb_completion_allowed,away_opp_cb_rating_allowed,div_game
0,1.0,0.8,16.4,8.0,203.4,1.2,0.2,4.174311,203.8,1.4,...,0,0.0,-3.0,1.0,118.7,0.0,-5.0,0.8034,100.46,0


In [95]:
import joblib

saved = joblib.load("../models/win_probability_model.joblib")
model = saved['model']
imputer = saved['imputer']
feature_cols = saved['feature_cols']

# Make sure our assembled row has exactly the columns the model expects, in the right order
missing_cols = set(feature_cols) - set(feature_row.columns)
extra_cols = set(feature_row.columns) - set(feature_cols)
print("Missing columns the model needs:", missing_cols)
print("Extra columns we built but don't need:", extra_cols)

Missing columns the model needs: set()
Extra columns we built but don't need: set()


In [96]:
X_live = feature_row[feature_cols]
X_live_imputed = pd.DataFrame(imputer.transform(X_live), columns=feature_cols)

home_win_prob = model.predict_proba(X_live_imputed)[0][1]

print(f"{away_team} @ {home_team}")
print(f"{home_team} win probability: {home_win_prob:.1%}")
print(f"{away_team} win probability: {1 - home_win_prob:.1%}")

NE @ SEA
SEA win probability: 72.6%
NE win probability: 27.4%


In [106]:
import os
print(os.getcwd())
print(os.path.exists("models/win_probability_model.joblib"))

c:\Users\drdre\Projects\sports_betting\notebooks
False


In [107]:
model_bundle = load_model(model_path="../models/win_probability_model.joblib")
snapshots = build_snapshots(db_path="../data/nfl.db")
predictions = predict_week(2026, 1, snapshots, model_bundle)

sched2026 = nfl.load_schedules(seasons=[2026]).to_pandas()

print(len(predictions))
predictions.head()

16


,game_id,home_team,away_team,gameday,home_win_prob,away_win_prob
0,2026_01_NE_SEA,SEA,NE,2026-09-09,0.726124,0.273876
1,2026_01_SF_LA,LA,SF,2026-09-10,0.619249,0.380751
2,2026_01_CHI_CAR,CAR,CHI,2026-09-13,0.326987,0.673013
3,2026_01_TB_CIN,CIN,TB,2026-09-13,0.657562,0.342438
4,2026_01_NO_DET,DET,NO,2026-09-13,0.773288,0.226712


In [109]:
def moneyline_to_prob(ml):
    if pd.isna(ml):
        return None
    if ml < 0:
        return -ml / (-ml + 100)
    else:
        return 100 / (ml + 100)

week1_odds = sched2026[sched2026['week']==1][['game_id', 'home_team', 'away_team', 'home_moneyline']].copy()
week1_odds['vegas_home_prob'] = week1_odds['home_moneyline'].apply(moneyline_to_prob)

comparison = predictions.merge(week1_odds[['game_id', 'vegas_home_prob']], on='game_id', how='left')
comparison['difference'] = comparison['home_win_prob'] - comparison['vegas_home_prob']

comparison[['home_team', 'away_team', 'home_win_prob', 'vegas_home_prob', 'difference']].sort_values('difference', key=abs, ascending=False)

,home_team,away_team,home_win_prob,vegas_home_prob,difference
11,LV,MIA,0.300253,0.672131,-0.371878
15,KC,DEN,0.278873,0.596774,-0.317901
10,LAC,ARI,0.641411,0.842520,-0.201109
2,CAR,CHI,0.326987,0.454545,-0.127559
13,PHI,WAS,0.796680,0.685535,0.111145
0,SEA,NE,0.726124,0.636364,0.089760
6,IND,BAL,0.325751,0.408163,-0.082413
7,JAX,CLE,0.827507,0.791667,0.035840
1,LA,SF,0.619249,0.649123,-0.029873
5,HOU,BUF,0.476379,0.504950,-0.028572


In [110]:
kc_elo = snapshots['current_elo'][snapshots['current_elo']['team']=='KC']
den_elo = snapshots['current_elo'][snapshots['current_elo']['team']=='DEN']
print(kc_elo, den_elo)

kc_qb = snapshots['current_qb'][snapshots['current_qb']['team']=='KC']
print(kc_qb)

   team  current_elo
20   KC  1514.127594   team  current_elo
3  DEN  1559.807097
     team  recent_passing_yards  recent_passing_tds  \
3435   KC                  88.0                 0.0   

      recent_passing_interceptions  recent_passing_epa  
3435                           0.0           -6.825473  


In [111]:
qb_check = pd.read_sql_query("SELECT * FROM games_with_features WHERE home_team='KC' OR away_team='KC'", 
                               __import__('sqlite3').connect("../data/nfl.db"))
print(qb_check[['gameday','home_team','away_team','home_qb_name','away_qb_name']].tail(6))

        gameday home_team away_team     home_qb_name     away_qb_name
200  2025-11-27       DAL        KC     Dak Prescott  Patrick Mahomes
201  2025-12-07        KC       HOU  Patrick Mahomes      C.J. Stroud
202  2025-12-14        KC       LAC  Patrick Mahomes   Justin Herbert
203  2025-12-21       TEN        KC         Cam Ward  Gardner Minshew
204  2025-12-25        KC       DEN   Chris Oladokun           Bo Nix
205  2026-01-04        LV        KC    Kenny Pickett   Chris Oladokun


In [112]:
depth = nfl.load_depth_charts(seasons=[2025]).to_pandas()

qb_depth = depth[(depth['pos_abb'] == 'QB') & (depth['pos_rank'] == 1)].copy()
qb_depth['dt'] = pd.to_datetime(qb_depth['dt'])

# Take the MOST RECENT depth chart snapshot per team (depth charts are updated daily)
current_qb_depth = qb_depth.loc[qb_depth.groupby('team')['dt'].idxmax()][['team', 'gsis_id', 'player_name']]

current_qb_depth[current_qb_depth['team'] == 'KC']

,team,gsis_id,player_name
1124,KC,00-0033873,Patrick Mahomes
